In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [2]:
data = pd.read_csv("Iris.csv")
data.head()

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


In [3]:
values = data["Species"].unique()
print(values)
species_map =dict()
ct=0
for i in values:
    species_map[i]=ct
    ct+=1
print(species_map)
data["Species"]= (
    data["Species"].astype(str)
    .str.strip()
    .map(species_map)
    .fillna(0)
)
data.head()

['Iris-setosa' 'Iris-versicolor' 'Iris-virginica']
{'Iris-setosa': 0, 'Iris-versicolor': 1, 'Iris-virginica': 2}


,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,0
1,2,4.9,3.0,1.4,0.2,0
2,3,4.7,3.2,1.3,0.2,0
3,4,4.6,3.1,1.5,0.2,0
4,5,5.0,3.6,1.4,0.2,0


In [4]:
data.drop("Id",axis=1,inplace = True)
data.head()

,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


In [5]:
y = data['Species']
X = data.drop( 'Species', axis=1)

In [6]:
X = X.to_numpy()
y= y.to_numpy()
X_train, X_test, y_train,y_test = train_test_split(
    X,y , test_size = 0.2
)

In [7]:
# making the node class 
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

        
    def is_leaf(self):
        return self.value is not None
        

In [ ]:
# making the decision tree class:
class DecisionTree():
    def __init__(self,minsize= 2,mdepth = 100, nfeats= None):
        self.minsize = minsize
        self.mdepth= mdepth
        self.nfeats = nfeats
        #the root node of the tree
        self.root = None
    def fit(self,X,y):
        #handling the case where nfeats is usually the no of features in X
        self.nfeats = X.shape[1] if self.nfeats is None else min(self.nfeats,X.shape[1])
        self.root = self._make(X,y)
    def _make(self,X,y,depth =0):
        #basic info
        m,n = X.shape
        k  = len(np.unique(y))
        #chekcing stopping criteria
        if(depth>=self.mdepth
          or k==1 or 
          m < self.minsize):
            leaf_value = self._frequent(y)
            return Node(value= leaf_value)
        #finding best split
        #first taking the random features 
        bfeature,bthresh = self._best_split(X,y,range(n))
        if bfeature is None:
            leaf_value = self._frequent(y)
            return Node(value=leaf_value)
        #create child nodes recursively 
        l_idxs = np.argwhere(X[:,bfeature]<=bthresh).flatten()
        r_idxs = np.argwhere(X[:,bfeature]>bthresh).flatten()
        if len(l_idxs) == 0 or len(r_idxs) == 0:
            return Node(value=self._frequent(y))
        left = self._make(X[l_idxs], y[l_idxs],depth+1)
        
        right = self._make(X[r_idxs], y[r_idxs],depth+1)
        #return node
        return Node(bfeature,bthresh,left,right)
        
    def _frequent(self,y):
        if len(y) == 0:
            return None
        values,freq= np.unique(y,return_counts = True)
        max_idx = np.argmax(freq)
        return values[max_idx]
    def _best_split(self,X,y,feat_idxs):
        bgain =-1
        idx ,thresh = None,None
        for feat_idx in feat_idxs:
            X_col = X[:,feat_idx]
            thresholds = np.unique(X_col)
            
            for thr in thresholds:
                gain = self._infog(y,X_col,thr)
                if(gain>bgain):
                    bgain = gain
                    idx = feat_idx
                    thresh = thr
        return idx,thresh
    def _infog(self,y,X_col,thr):
        #getting the parent ent 
        prnt_ent = self._ent(y)
        #create the children
        l_idxs = np.argwhere(X_col<=thr).flatten()
        r_idxs = np.argwhere(X_col>thr).flatten()
        
        n = len(y)
        n_l , n_r = len(l_idxs), len(r_idxs)
        if(n_l==0 or n_r==0):
            return 0
        
        e_l , e_r = self._ent(y[l_idxs]),self._ent(y[r_idxs])
        chld_ent = n_l/n*e_l + n_r/n*e_r
        #calcualte the gain 
        info_gain = prnt_ent - chld_ent
        #return 
        return info_gain
    
    def _ent(self,y):
        hist = np.bincount(y)
        props = hist/len(y)
        error_handler = 1e-15
        return -np.sum([p*np.log(p+error_handler) for p in props])
    
    
    def predict(self,X):
        return np.array([self._getpred(x,self.root) for x in X])
    def _getpred(self,X,node):
        if node.is_leaf():
            return node.value
        if X[node.feature] <= node.threshold:
            return self._getpred(X, node.left)  
        else:
            return self._getpred(X, node.right)
                            

In [30]:
model = DecisionTree()
model.fit(X_train,y_train)
y_pred =model.predict(X_test)
print(f"the accuray is {np.mean(y_pred==y_test):.2f}%")

the accuray is 0.93%
